In [1]:
%matplotlib inline
import os
import torch
import torchvision
from d2l import torch as d2l

In [ ]:
def rand_crop(feature, label, height, width):
    """随机裁剪特征和标签图像进行数据增强""" 
    rect = torchvision.transforms.RandomCrop.get_params(feature, (height, width))  # 在同一随机位置裁剪，保证feature和label能够对上位置
    feature = torchvision.transforms.functional.crop(feature, *rect)
    label = torchvision.transforms.functional.crop(label, *rect)
    return feature, label

## 语义分割怎么实现

**核心思路**：分类网络（如 ResNet）最后是全连接层，输出一个类别编号。语义分割把全连接层**换成 1×1 卷积层**，输出不再是1个数字，而是一张 H×W 的图，每个像素是一个类别预测。

具体来说，CNN 分类网络前半部分通过卷积+池化不断缩小特征图、提取高层语义；语义分割需要把这个缩小的特征图**还原到原图大小**，所以后半部分用转置卷积或双线性插值上采样。整张图进、整张图出，每个像素一个类别。</cell_id>
<｜｜DSML｜｜parameter name="cell_type" string="true">markdown

In [ ]:
# 读取语义分割数据集（以 Pascal VOC 2012 为例，d2l 提供了封装好的下载和读取函数）
# 数据集包含输入图像和对应的标签图像（每个像素标注类别编号）
# crop_size 统一裁成 320×480，batch_size 设为 32
batch_size, crop_size = 32, (320, 480)
train_iter, test_iter = d2l.load_data_voc(batch_size, crop_size)

In [ ]:
# 语义分割模型：全卷积网络 (Fully Convolutional Network, FCN)
# 
# 结构：卷积基干网络（ResNet-18 去掉最后的全局池化和全连接层）→ 1×1 卷积层 → 转置卷积上采样
# 
# 卷积基干：提取高层语义特征，特征图尺寸会缩小（步幅为2的卷积/池化每次缩一半）
# 1×1 卷积：通道从512变成类别数（21类 = 20个物体 + 1个背景），每个位置一个类别预测
# 转置卷积：把缩小的特征图放大回原图（学名叫"上采样"），卷积核大小=缩放倍数，步幅=缩放倍数
pretrained_net = torchvision.models.resnet18(pretrained=True)
# 取 ResNet 的前4层（去掉最后的全局平均池化 avgpool 和全连接层 fc）
net = torch.nn.Sequential(*list(pretrained_net.children())[:-2])

# 把分类头换成语义分割头
num_classes = 21  # VOC 数据集：20类物体 + 1类背景
net.add_module('final_conv', torch.nn.Conv2d(512, num_classes, kernel_size=1))       # 1×1卷积，不改变空间尺寸，只把通道数变成类别数
net.add_module('transpose_conv', torch.nn.ConvTranspose2d(num_classes, num_classes,
    kernel_size=64, padding=16, stride=32))  # 转置卷积上采样32倍（ResNet-18缩了32倍，所以要放大回去）

In [ ]:
# 初始化转置卷积的权重为双线性插值核
# 这样训练初期就不是随机上采样，而是用双线性插值——收敛更快
def bilinear_kernel(in_channels, out_channels, kernel_size):
    """构造双线性插值的卷积核权重"""
    factor = (kernel_size + 1) // 2  # 求核的半宽
    if kernel_size % 2 == 1:  # 求核的中心
        center = factor - 1  
    else:
        center = factor - 0.5
    og = (torch.arange(kernel_size).reshape(-1, 1),
          torch.arange(kernel_size).reshape(1, -1))  # 利用广播机制构造新的初始化核
    filt = (1 - torch.abs(og[0] - center) / factor) * \
           (1 - torch.abs(og[1] - center) / factor)  # 求每个位置的权重
    weight = torch.zeros((in_channels, out_channels, kernel_size, kernel_size))
    weight[range(in_channels), range(out_channels), :, :] = filt
    return weight

# 用双线性插值初始化转置卷积，而不是随机权重
net.transpose_conv.weight.data.copy_(bilinear_kernel(num_classes, num_classes, 64))

In [ ]:
# 语义分割的损失函数：逐像素交叉熵
# 
# 分类任务：一张图一个标签 → 一个交叉熵损失
# 语义分割：一张图 H×W 个像素，每个像素一个标签 → H×W 个交叉熵损失，求平均
# 
# torch.nn.CrossEntropyLoss 输入形状：
#   - 预测：(batch_size, num_classes, H, W)  ← 注意通道维是类别概率
#   - 标签：(batch_size, H, W)               ← 每个像素的类别编号（整数）
# 内部自动把 (N,C,H,W) 展平成 (N*H*W, C) 和 (N*H*W,) 来计算
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
# 训练
# 训练过程与分类任务类似，只是每轮也跑验证集
# d2l 的 train_batch_ch13 内部：前向 → loss → 反向 → 更新参数
# 预测输出形状：(batch, num_classes, H, W)，标签形状：(batch, H, W)
lr, num_epochs, devices = 0.001, 5, d2l.try_all_gpus()
trainer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=0.001)
d2l.train_ch13(net, train_iter, test_iter, loss_fn, trainer, num_epochs, devices)

In [ ]:
# 预测可视化
# 取一张测试图，过一遍网络，看分割效果
# predict 的输出是 (batch, num_classes, H, W)，取 argmax 得到每个像素的预测类别
def predict(img):
    """对单张图片做语义分割预测，返回类别图"""
    net.eval()
    X = test_iter.dataset.normalize_image(img).unsqueeze(0)  # 归一化 + 加 batch 维
    with torch.no_grad():
        pred = net(X.to(devices[0])).squeeze(0)  # (C, H, W)
    return pred.argmax(dim=0)  # (H, W)，每个位置取最大概率的类别编号

# 随机看几张测试集的预测结果
imgs, labels = next(iter(test_iter))  # 取一个 batch
imgs, labels = imgs[:4], labels[:4]   # 只看前4张
preds = predict(imgs[0]).unsqueeze(0)
for i in range(1, imgs.shape[0]):
    preds = torch.cat((preds, predict(imgs[i]).unsqueeze(0)), dim=0)

# 显示：原图 | 预测分割图 | 真实标签
d2l.show_images([imgs, preds.unsqueeze(1).float() / num_classes, labels.unsqueeze(1).float() / num_classes],
                rows=1, titles=['输入图片', '预测分割', '真实标签'])